# 04 · Preprocessing — from clean beats to model-ready features

Everything here follows from the EDA:

| From notebooks 02–03 | Decision taken here |
|---|---|
| No nulls, NaNs, duplicates or leakage in MIT-BIH | No cleaning stage |
| 113:1 imbalance | Balanced class weights; resampling available but training-split only |
| Published MIT-BIH split already stratified | Test split untouched; validation carved out of train, stratified |
| PTB ships no split | Stratified 70/15/15 created reproducibly |
| Descriptors collinear and weakly linked to the label | 7 descriptors kept, waveform stays the primary input |
| 33–50% of each vector is padding | Descriptors computed on the unpadded prefix |

The result is a `pyspark.ml.Pipeline` — savable, reloadable, and identical on a
laptop and on Databricks.

In [1]:
import sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# Works from a fresh clone (no `pip install -e .` needed) and on Databricks,
# where the repo folder is added to the workspace instead of installed.
try:
    import ecg
except ModuleNotFoundError:
    sys.path.insert(0, str(Path.cwd().parent / "src"))
    import ecg

print("ecg", ecg.__version__)

ecg 0.1.0


In [2]:
import pandas as pd
from pyspark.sql import functions as F

from ecg import get_spark, load_config
from ecg.features import MODEL_FEATURE_COLUMNS
from ecg.ingest import load_beats
from ecg import preprocessing as pp

pd.set_option("display.width", 170)

cfg = load_config()
spark = get_spark(cfg)
spark.sparkContext.setLogLevel("ERROR")

beats = load_beats(spark, cfg).cache()
print(f"{beats.count():,} beats")
MODEL_FEATURE_COLUMNS

123,998 beats


['signal_length',
 'amp_min',
 'amp_mean',
 'amp_std',
 'peak_index',
 'mean_abs_diff',
 'max_abs_diff']

## 1 · Splits

`randomSplit` would be wrong here. With class `F` at 0.73% of the data, an
unstratified 15% validation slice can easily land 20% away from the true class
proportion. `stratified_split` instead ranks each class independently by a seeded
random order and cuts at the requested percentiles, so the proportions are exact
and the assignment is reproducible.

In [3]:
mitbih = beats.where(F.col("source") == "mitbih")
mitbih_split = pp.holdout_validation(
    mitbih, val_fraction=cfg.val_fraction, output_col="split_final", seed=cfg.seed
).cache()

summary = pp.split_summary(mitbih_split)
summary.pivot(index="label_name", columns="split_final", values="pct").loc[["N", "S", "V", "F", "Q"]]

split_final,test,train,val
label_name,,,
N,82.761,82.776,82.757
S,2.540,2.538,2.543
V,6.614,6.610,6.615
F,0.740,0.731,0.738
Q,7.345,7.345,7.346


In [4]:
summary

,split_final,label,label_name,n_beats,pct
0,test,0,N,18118,82.761
1,test,1,S,556,2.540
2,test,2,V,1448,6.614
3,test,3,F,162,0.740
4,test,4,Q,1608,7.345
5,train,0,N,61600,82.776
6,train,1,S,1889,2.538
7,train,2,V,4919,6.610
8,train,3,F,544,0.731
9,train,4,Q,5466,7.345


**Check.** Each class holds the same share in `train`, `val` and `test` to within a
few hundredths of a point, and the `test` column is the published split, untouched.

In [5]:
ptb = beats.where(F.col("source") == "ptbdb")
ptb_split = pp.stratified_split(ptb, fractions=pp.DEFAULT_SPLIT_FRACTIONS, output_col="split_final", seed=cfg.seed)
pp.split_summary(ptb_split)

,split_final,label,label_name,n_beats,pct
0,test,0,normal,607,27.806
1,test,1,abnormal,1576,72.194
2,train,0,normal,2832,27.803
3,train,1,abnormal,7354,72.197
4,val,0,normal,607,27.806
5,val,1,abnormal,1576,72.194


## 2 · Handling the imbalance

Two options, in order of preference.

**Class weights** change no data. Every Spark ML classifier accepts a `weightCol`,
and the balanced formula `n / (k · n_class)` is the same one scikit-learn uses, so
the weights transfer to any framework the modelling phase settles on.

In [6]:
train = mitbih_split.where(F.col("split_final") == "train")
weights = pp.class_weights(train)
pd.DataFrame(
    [
        {"label": label, "n_beats": pp.class_counts(train)[label], "weight": round(weight, 3)}
        for label, weight in sorted(weights.items())
    ]
)

,label,n_beats,weight
0,0,61600,0.242
1,1,1889,7.879
2,2,4919,3.026
3,3,544,27.360
4,4,5466,2.723


**Resampling** changes the data, and is shown here so the trade-off is explicit.
Undersampling to the minority shrinks the training split from 74,418 beats to about
2,700 — a 96% loss, most of it normal beats a model still needs to learn well.
Oversampling replicates the rare beats up to the majority size instead, at the cost
of near-duplicate rows that invite overfitting. (Both use Bernoulli sampling, so the
resulting counts land near the target rather than exactly on it.)

Either way this is a **training-split-only** operation: rebalancing a validation or
test set changes the prior and makes every metric optimistic.

In [7]:
comparison = pd.DataFrame(
    {
        "original": pp.class_counts(train),
        "undersampled": pp.class_counts(pp.resample(train, "undersample", seed=cfg.seed)),
        "oversampled": pp.class_counts(pp.resample(train, "oversample", seed=cfg.seed)),
    }
).sort_index()
comparison.index.name = "label"
comparison

,original,undersampled,oversampled
label,,,
0,61600,501,61600
1,1889,575,61616
2,4919,497,61635
3,544,544,61599
4,5466,570,61574


## 3 · The pipeline

Four stages, two of them custom transformers so the feature definitions travel
inside the saved model rather than living in a notebook cell:

1. `BeatFeatureTransformer` — the per-beat descriptors, computed on the unpadded prefix
2. `ArrayToVector` — the 187 samples become a dense MLlib vector
3. `VectorAssembler` — waveform ⊕ 7 descriptors → 194 dimensions
4. `StandardScaler` — zero mean, unit variance

The scaler is fit **on the training split only**. Fitting it on the full frame would
push validation and test statistics into the features — a subtle leak that inflates
every downstream score.

In [8]:
pipeline = pp.build_preprocessing_pipeline(scaler="standard")
[type(stage).__name__ for stage in pipeline.getStages()]

['BeatFeatureTransformer',
 'ArrayToVector',
 'VectorAssembler',
 'StandardScaler']

In [9]:
dataset, model = pp.build_dataset(beats, cfg, source="mitbih", scaler="standard", write=True)
row = dataset.select("beat_id", "label_name", "split_final", "class_weight", "features").first()
print("feature vector:", row["features"].size, "dimensions",
      f"({187} waveform + {len(MODEL_FEATURE_COLUMNS)} descriptors)")
print("class weight  :", round(row["class_weight"], 3))
print("first 6 values:", [round(v, 3) for v in row["features"].toArray()[:6]])

feature vector: 194 dimensions (187 waveform + 7 descriptors)
class weight  : 0.242
first 6 values: [np.float64(0.455), np.float64(0.021), np.float64(-0.661), np.float64(-0.155), np.float64(0.029), np.float64(-0.052)]


In [10]:
pp.split_summary(dataset)

,split_final,label,label_name,n_beats,pct
0,test,0,N,18118,82.761
1,test,1,S,556,2.540
2,test,2,V,1448,6.614
3,test,3,F,162,0.740
4,test,4,Q,1608,7.345
5,train,0,N,61600,82.776
6,train,1,S,1889,2.538
7,train,2,V,4919,6.610
8,train,3,F,544,0.731
9,train,4,Q,5466,7.345


## 4 · The same pipeline on PTB

In [11]:
ptb_dataset, ptb_model = pp.build_dataset(beats, cfg, source="ptbdb", scaler="standard", write=True)
pp.split_summary(ptb_dataset)

,split_final,label,label_name,n_beats,pct
0,test,0,normal,607,27.806
1,test,1,abnormal,1576,72.194
2,train,0,normal,2832,27.803
3,train,1,abnormal,7354,72.197
4,val,0,normal,607,27.806
5,val,1,abnormal,1576,72.194


## 5 · Reload check

A pipeline that cannot be reloaded is a pipeline that will drift. Reading it back
and re-transforming a handful of rows proves the saved artefact is self-contained,
custom stages included.

In [12]:
reloaded = pp.load_pipeline(cfg, "mitbih")
print([type(stage).__name__ for stage in reloaded.stages])

check = reloaded.transform(beats.where(F.col("source") == "mitbih").limit(5))
check.select("beat_id", "label_name", "signal_length", "amp_mean").show(truncate=False)
print("reloaded vector size:", check.select("features").first()[0].size)

['BeatFeatureTransformer', 'ArrayToVector', 'VectorAssembler', 'StandardScalerModel']


+-------------------------+----------+-------------+-------------------+
|beat_id                  |label_name|signal_length|amp_mean           |
+-------------------------+----------+-------------+-------------------+
|mitbih_train_042949672960|N         |126          |0.6530292898653045 |
|mitbih_train_042949672961|N         |118          |0.4739183582428653 |
|mitbih_train_042949672962|N         |153          |0.16526255642179571|
|mitbih_train_042949672963|N         |143          |0.14182563053144442|
|mitbih_train_042949672964|N         |125          |0.30094043793901804|
+-------------------------+----------+-------------+-------------------+



reloaded vector size: 194


## What is on disk now

```
data/processed/
├── beats/                            123,998 canonical beats, partitioned by source/split
├── features_mitbih/                  109,446 beats · 194-dim vectors · train/val/test
├── features_ptbdb/                    14,552 beats · 194-dim vectors · train/val/test
├── preprocessing_pipeline_mitbih/    fitted PipelineModel
└── preprocessing_pipeline_ptbdb/     fitted PipelineModel
```

Each row carries `features` (scaled vector), `label`, `label_name`, `split_final`
and `class_weight` — everything a classifier needs, and nothing it should not see.

**Next step (out of scope for this phase):** modelling. The natural sequence is a
Spark ML baseline on these vectors — logistic regression and random forest with
`weightCol="class_weight"`, scored by macro-F1 — followed by a 1-D convolutional
network on the raw 187-sample waveform, which is where the source paper's results
come from, and then the MIT-BIH → PTB transfer experiment.

In [13]:
beats.unpersist()
spark.stop()